In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import toml
from pathlib import Path
import util
import sys

sys.path.append("../../notebook_styling")
import psrc_theme

In [2]:


config = toml.load(Path(Path.cwd(), '..\..\..\..\configuration', 'validation_configuration.toml'))
input_config = toml.load(Path(Path.cwd(), '..\..\..\..\configuration', 'input_configuration.toml'))

data = util.ValidationData(config,input_config,['hh', 'person', 'tour', 'land_use', 'parcel_geog'])

In [3]:
hh = data.hh.join(data.land_use, how="left",left_on='hhno',right_on='parcelid').\
    join(data.parcel_geog, how="left",left_on='hhparcel',right_on='ParcelID').\
        to_pandas()
person = data.person.to_pandas()
tour = data.tour.to_pandas()

In [4]:
df_tour = tour.copy()
df_hh = hh.copy()
df_person = person.copy()

# auto_ownership with 4+
df_hh['auto_ownership_4+'] = df_hh['hhvehs'].apply(lambda x: "4+" if x>=4.0 else str(x))
# hhsize with 4+
df_hh['hhsize_4+'] = df_hh['hhsize'].apply(lambda x: "4+" if x>=4.0 else str(x))
# Add column for (potential) drivers adults (all hh members 16 and above)
df_hh['drivers'] = df_hh['hhsize']-df_hh['hh515']-df_hh['hhcu5']#-df_hh['hhhsc']
# auto availability
df_hh['auto_count_driver'] = df_hh['hhvehs']-df_hh['drivers']
df_hh['auto_available_driver'] = np.where(df_hh['drivers']<=0, "no driver",
                                          np.where(df_hh['hhvehs']<=0, "no car",
                                                   np.where(df_hh['auto_count_driver']<0, "cars fewer than drivers", "enough cars")))

# add person type labels
ptype_cat = {1: "full time worker",
             2: "part time worker",
             3: "non-worker age 65+",
             4: "other non-working adult",
             5: "university student",
             6: "grade school student/child age 16+",
             7: "child age 5-15",
             8: "child age 0-4"}
df_person['pptyp_label'] = df_person['pptyp'].map(ptype_cat)

mode_cat = {1: "1: walk",
            2: "2: bike",
            3: "3: sov",
            4: "4: hov 2",
            5: "5: hov 3+",
            6: "6: walk to transit",
            7: "7: park-and-ride",
            8: "8: school bus",
            9: "9: tnc"}
df_tour['tmodetp_label'] = df_tour['tmodetp'].map(mode_cat)

pdpurp_cat = {1: "Work",
              2: "School",
              3: "Escort",
              4: "Personal Business",
              5: "Shop",
              6: "Meal",
              7: "Social"}
df_tour['pdpurp_label'] = df_tour['pdpurp'].map(pdpurp_cat)

df_person = df_person[['pno','hhno','source','pptyp_label']].merge(df_hh[['hhno','source','auto_ownership_4+','hhsize_4+','drivers','auto_count_driver','auto_available_driver','CountyName']],
                          how='left', on=['hhno','source']) # get auto ownership from hh data

df_tour = df_tour.merge(df_person, how='left', on=['pno','hhno','source'])

In [5]:
hb_tour = df_tour.loc[(df_tour['parent']==0) & ~(df_tour['pdpurp'].isin([1,2,3]))].copy()

In [6]:
df_plot = hb_tour.groupby(['source','tmodetp_label'])['toexpfac'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['toexpfac']. \
    apply(lambda x: x / float(x.sum()))

df_plot_ct = hb_tour.groupby(['source','tmodetp_label'])['toexpfac'].count().reset_index(). \
    rename(columns={'toexpfac':'sample count'})
df_plot = df_plot.merge(df_plot_ct, on=['source','tmodetp_label'])

fig = px.bar(df_plot.sort_values(by=['source']), x="tmodetp_label", y="percentage", color="source",
             barmode="group",hover_data=['sample count'],title="other home-based tour mode")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  xaxis = dict(dtick = 1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".2%"))
fig.show()

In [7]:
df_plot = hb_tour.groupby(['source','pdpurp_label','tmodetp_label'])['toexpfac'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source','pdpurp_label'], group_keys=False)['toexpfac']. \
    apply(lambda x: x / float(x.sum()))

df_plot_ct = hb_tour.groupby(['source','pdpurp_label','tmodetp_label'])['toexpfac'].count().reset_index(). \
    rename(columns={'toexpfac':'sample count'})
df_plot = df_plot.merge(df_plot_ct, on=['source','pdpurp_label','tmodetp_label'])

fig = px.bar(df_plot.sort_values(['source','tmodetp_label']),
             x="percentage", y="tmodetp_label", color="source",barmode="group",
             facet_col='pdpurp_label', facet_col_wrap=1, orientation='h',
             hover_data=['sample count'],
             category_orders={"tmodetp_label":["1: walk","2: bike","3: sov","4: hov 2","5: hov 3+","6: walk to transit",
                                               "7: park-and-ride","8: school bus","9: otherÃ¢â‚¬â€œsurvey only"]},
             title="other home-based tour mode choice")
fig.update_layout(height=1000, width=650)
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
fig.show()

### mode choice by segment

In [8]:
def plot_mode_choice(df: pd.DataFrame, grp_var: str, order_list: dict, title_name: str, n_nol: int, height=400, width=800):
    df_plot = df.groupby(['source',grp_var,'tmodetp_label'])['toexpfac'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',grp_var], group_keys=False)['toexpfac']. \
        apply(lambda x: x / float(x.sum()))

    df_plot_ct = df.groupby(['source',grp_var,'tmodetp_label'])['toexpfac'].count().reset_index(). \
        rename(columns={'toexpfac':'sample count'})
    df_plot = df_plot.merge(df_plot_ct, on=['source',grp_var,'tmodetp_label'])

    fig = px.bar(df_plot.sort_values(['tmodetp_label']),
                 x="percentage", y="tmodetp_label", color="source",barmode="group",
                 facet_col=grp_var, facet_col_wrap=n_nol, orientation='h',
                 hover_data=['sample count'],
                 category_orders=order_list,
                 title="other home-based tour mode choice by " + title_name)
    fig.update_layout(height=height, width=width)
    fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
    fig.for_each_xaxis(lambda a: a.update(tickformat = ".1%"))
    fig.show()

In [9]:
incl_county = ["King", "Kitsap", "Pierce", "Snohomish"]

plot_mode_choice(hb_tour.loc[hb_tour["CountyName"].isin(incl_county)],"CountyName",
                 {"CountyName":["King", "Kitsap", "Pierce", "Snohomish"],
                  "tmodetp_label":mode_cat.values()},
                 "home county",
                 2,600)

In [10]:
plot_mode_choice(hb_tour,"pptyp_label",
                 {"pptyp_label":["full time worker","part time worker","non-worker age 65+","other non-working adult",
                                 "university student","grade school student/child age 16+","child age 5-15","child age 0-4"],
                  "tmodetp_label":mode_cat.values()},
                 "person type",2,1000)

In [11]:
plot_mode_choice(hb_tour,"hhsize_4+",
                 {"hhsize_4+":["1","2","3","4+"],
                  "tmodetp_label":mode_cat.values()},
                 "household size",2,600)

In [12]:
plot_mode_choice(hb_tour.loc[hb_tour['auto_ownership_4+']!="-1"],"auto_ownership_4+",
                 {"auto_ownership_4+":["0","1","2","3","4+"],
                  "tmodetp_label":mode_cat.values()},
                 "auto ownership",2,800)

In [13]:
plot_mode_choice(hb_tour.loc[hb_tour['auto_available_driver'] != "no dirver"], "auto_available_driver",
                 {"auto_available_driver": ["no car", "cars fewer than drivers", "enough cars"],
                  "tmodetp_label": mode_cat.values()},
                 "auto availability (driver, showing only households with at least one driver)", 3, 500, 1000)